# Gorham, Maine — Land Value Tax Model (4:1 Split-Rate)

Revenue-neutral split-rate LVT model for the **Town of Gorham, Cumberland County, Maine**
(FIPS 23005). Models the full town property-tax commitment (municipal + school + county are
billed as one **14.800-mill** rate) shifted to a **4:1 split-rate** (land taxed at 4× the
building rate), preserving all existing exemptions (Homestead, veteran, etc.).

**Legality note:** Maine is a "just value" state with no municipal classification/split-rate
authority — a real split-rate LVT would require a state constitutional amendment (LD 1464's
statutory attempt died in 2025). This model is an empirical *what-if*, not a near-term pathway.

## Data sources (see `data/` helper scripts for provenance)
| Layer | Source | Role |
|---|---|---|
| Per-parcel **values, exemptions, homestead** | **CCRA 2025-2026 Real Estate Tax Commitment Book** (PDF, committed 09/29/2025 @ 14.800) parsed by `data/parse_commitment.py` | Authoritative, current, complete roll (7,504 parcels) |
| **Use codes** (classification) | **Vision Government Solutions** CAMA (`gis.vgsi.com/gorhamme`) scraped by `data/scrape_vgsi.py` | Maine use codes, ~66% of parcels |
| **Geometry** | **Maine Parcels Organized Towns** statewide layer, fetched by `data/fetch_geometry.py` | Parcel polygons for the map (~82%) |

The Maine statewide ADB was rejected as a value source: it carries *no* use codes for Gorham and
its values are a stale pre-2022-revaluation vintage (~half current). The commitment book is the
town's own certified roll and prints grand totals used for validation below.

## Policy decisions
- **Levy:** full town commitment (14.800 mills, all overlapping levies) — Gorham bills one rate.
- **Reform:** 4:1 split-rate, revenue-neutral.
- **Exemptions:** preserved (the commitment "Assessment" is already net of Homestead/veteran/etc.).

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'gorham'
STATE_FIPS = '23'
COUNTY_FIPS = '005'
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0
MILL_RATE = 14.800          # FY2025-26 combined commitment rate (per $1,000)

# Official anchors from the CCRA 2025-2026 Real Estate Commitment Book grand totals
BOOK_LAND       = 783_259_400
BOOK_BUILDING   = 2_520_461_600
BOOK_EXEMPTION  = 402_032_800
BOOK_TAXABLE    = 2_901_688_200      # total taxable valuation (net of exemption)
BOOK_COMMITMENT = 42_944_985.36      # total real-estate tax committed

# Parcel-map identity columns
PARCEL_ID_COL = 'account'            # map/lot account number
OWNER_NAME_COL = 'name'
OWNER_ADDRESS_COL = 'PROP_LOC'
PARCEL_URL_TEMPLATE = None           # VGSI links use an internal PID, not the account/maplot

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load cached parcel data

The three sources are prepared by the helper scripts in `data/` and cached as parquet.
Regenerate any missing cache by running (from the repo root):

```bash
python3 cities/gorham/scrape_vgsi.py       # VGSI use codes  -> data/vgsi.parquet
python3 cities/gorham/parse_commitment.py  # commitment roll -> data/commitment.parquet
python3 cities/gorham/fetch_geometry.py    # parcel polygons -> data/parcels.gpq
```

In [ ]:
def _require(path, script):
    p = DATA_DIR / path
    if not p.exists():
        raise FileNotFoundError(
            f"{p} missing. Regenerate with: python3 cities/{CITY_NAME}/{script} (run from repo root)")
    return p

com = pd.read_parquet(_require('commitment.parquet', 'parse_commitment.py'))
vg  = pd.read_parquet(_require('vgsi.parquet', 'scrape_vgsi.py'))
geo = gpd.read_parquet(_require('parcels.gpq', 'fetch_geometry.py'))

com['maplot'] = com['maplot'].str.upper()
print(f"Commitment parcels: {len(com):,}  (taxable {int((com.assessment>0).sum()):,}, "
      f"full-exempt {int((com.assessment<=0).sum()):,})")
print(f"VGSI use-code rows: {len(vg):,}")
print(f"Geometry polygons:  {len(geo):,}")

# Validate the commitment parse against the book's printed grand totals
print("\nCommitment parse vs book grand totals:")
for label, got, book in [('Land', com.land.sum(), BOOK_LAND),
                          ('Building', com.building.sum(), BOOK_BUILDING),
                          ('Exemption', com.exemption.sum(), BOOK_EXEMPTION),
                          ('Assessment', com.assessment.sum(), BOOK_TAXABLE)]:
    print(f"  {label:11} parsed ${got:>15,.0f}  book ${book:>15,.0f}  ({got/book*100-100:+.2f}%)")

## Section 3 — Join use codes and classify

VGSI Maine use codes are joined to the commitment roll on the normalized map/lot key. Where a
parcel is outside VGSI's public coverage (~34%), the category is inferred from the commitment
book: `$0 building` → Vacant Land; Homestead/veteran flag → Single Family Residential; otherwise
Other Residential (a documented residual — see limitations).

| Concept | Column | Notes |
|---|---|---|
| Land value (gross) | `land` | reconstructed = assessment + exemption − building |
| Building value (gross) | `building` | |
| Exemption ($) | `exemption` | Homestead $22,500 (=$25k × 90% ratio), veteran, etc. |
| Taxable total | `assessment` | book "Assessment" = land + building − exemption |
| Use code | `use_code` | VGSI Maine code (1010=SFR, 1030=mobile home, 1020=condo, 3/4=comm/ind, 7=current-use) |
| Full-exempt flag | `full_exmp` | assessment == 0 |
| Account / map-lot | `account` / `maplot` | join key to geometry |

In [ ]:
def norm_vgsi(mblu):
    parts = [p.strip() for p in str(mblu).split('/') if p.strip()]
    return '-'.join(parts).upper() if parts else None

vg['maplot'] = vg['mblu'].map(norm_vgsi)
com = com.merge(
    vg[['maplot', 'use_code', 'use_desc']].dropna(subset=['maplot']).drop_duplicates('maplot'),
    on='maplot', how='left')
print(f"VGSI use code coverage: {com.use_code.notna().mean()*100:.1f}% of parcels")


def cat_from_use(uc):
    if uc is None or (isinstance(uc, float) and np.isnan(uc)):
        return None
    uc = str(uc); d = uc[:1]
    if uc in ('1010', '1011', '1012', '1013', '1014', '1015', '1025'):
        return 'Single Family Residential'
    if uc == '1020':
        return 'Condominium'
    if uc in ('1040', '1050', '1090'):
        return 'Small Multi-Family (2-4 units)'
    if uc.startswith(('111', '112', '113')):
        return 'Large Multi-Family (5+ units)'
    if uc == '1030':
        return 'Mobile Home'                       # often building-only on leased park land
    if uc in ('1300', '1340'):
        return 'Vacant Land'
    if d == '1':
        return 'Other Residential'
    if uc in ('3400', '3401'):
        return 'Office / Commercial Condo'
    if uc.startswith(('324', '325')):
        return 'Retail / General Commercial'
    if uc == '3040':
        return 'Other Commercial'
    if d == '3':
        return 'Retail / General Commercial'
    if uc.startswith(('422', '423')):
        return 'Other'                             # utility ROW / dams / transmission land
    if d == '4':
        return 'Industrial'
    if d == '7':
        return 'Agricultural'                      # current-use forest/farm/open land
    return 'Other'

com['PROPERTY_CATEGORY'] = com.use_code.map(cat_from_use)

# Inference for parcels lacking a VGSI use code
_vet = com.exempt_desc.fillna('').str.contains('Vet|Widow|Blind|Paraplegic|Homestead',
                                               case=False, regex=True)
com.loc[com.PROPERTY_CATEGORY.isna() & (com.building <= 0), 'PROPERTY_CATEGORY'] = 'Vacant Land'
com.loc[com.PROPERTY_CATEGORY.isna() & (com.homestead | _vet),
        'PROPERTY_CATEGORY'] = 'Single Family Residential'
com.loc[com.PROPERTY_CATEGORY.isna(), 'PROPERTY_CATEGORY'] = 'Other Residential'

# Override: any $0-building parcel -> Vacant Land
com.loc[com.building <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(com['PROPERTY_CATEGORY'].value_counts())

## Section 4 — Rebuild the current tax and validate (Gate 1)

Current tax = taxable **Assessment** × 14.800 / 1000. The commitment "Assessment" is already net
of exemptions, so this reproduces the actual bill. The exemption is split off the building first,
then the land (standard hierarchy), giving the taxable land/building components the split-rate
solver needs.

In [ ]:
# Exemption hierarchy: improvement first, then land
com['taxable_improvement_value'] = (com.building - com.exemption).clip(lower=0)
_rem_ex = (com.exemption - com.building).clip(lower=0)
com['taxable_land_value'] = (com.land - _rem_ex).clip(lower=0)
com['current_tax'] = com.assessment * MILL_RATE / 1000.0
com['full_exmp'] = (com.assessment <= 0).astype(int)

# consistency: taxable land + improvement must equal the book assessment
assert (abs((com.taxable_land_value + com.taxable_improvement_value) - com.assessment).max()) < 1e-6

modeled_rev = com.current_tax.sum()
gap = (modeled_rev / BOOK_COMMITMENT - 1) * 100
print(f"Modeled current tax: ${modeled_rev:,.0f}")
print(f"Book commitment:     ${BOOK_COMMITMENT:,.0f}")
print(f"Gap: {gap:+.2f}%  (residual = commitment-book PDF condo/interleaved parse; see limitations)")
assert abs(gap) < 5.0

## Section 5 — 4:1 split-rate model

Fully-exempt parcels (assessment == 0, e.g. Town of Gorham, University of Southern Maine, churches)
are held out of the revenue-neutral solver and excluded from the export and charts; they are
reported by count only.

In [ ]:
n_exempt = int((com['full_exmp'] == 1).sum())
model_df = com[com['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, model_df = model_split_rate_tax(
    df=model_df,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=model_df['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, charts).")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}"
      f"   (ratio {land_millage/improvement_millage:.2f}:1)")
print(f"Revenue check: ${new_revenue:,.0f}  (neutral vs ${model_df['current_tax'].sum():,.0f})")

category_summary = calculate_category_tax_summary(
    df=model_df, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax')
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

### Gate 5 — artifact scan

Reading the results against economic priors. Building shares are plausible by category (no
built-up type pinned at ~0% building), there is no ceiling clustering (distinct categories have
distinct medians), and directions are sensible: **Vacant Land** rises sharply (+127% median),
**Mobile Homes** (building-only on leased land) fall most (−43%), building-heavy **Condominiums**
fall, and large-lot **Single-Family Residential** sits near the split-rate break-even (~+3.5%
median) — a genuine feature of a low-density suburban/rural town, not a data artifact.

In [ ]:
d = model_df.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3).to_string())

## Section 6 — Exploration chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
model_df.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.axvline(0, color='k', lw=0.8)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census join + export + report + parcel map

The model runs on all taxable parcels; the interactive map shows the ~82% with parcel geometry
(condominium units share a single building footprint and are not individually mappable). Census
demographics are joined to the geometry-bearing parcels and merged back by account.

In [ ]:
# Attach model results to geometry (right join keeps all taxable parcels; ~18% lack geometry)
gdf = geo[['maplot', 'geometry', 'PROP_LOC', 'STATE_ID']].merge(model_df, on='maplot', how='right')
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
print(f"Modeled parcels: {len(gdf):,};  with geometry (mappable): {int(gdf.geometry.notna().sum()):,}")

In [ ]:
# Census join — OPTIONAL, on the geometry-bearing subset, merged back by account
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        _sub = gdf[gdf.geometry.notna()].copy()
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            census_data, census_gdf = _future.result(timeout=90)
        _sub = match_to_census_blockgroups(_sub, census_gdf)
        if 'minority_pct' not in _sub.columns and {'total_pop', 'white_pop'} <= set(_sub.columns):
            _sub['minority_pct'] = ((_sub.total_pop - _sub.white_pop) / _sub.total_pop * 100).round(2)
        if 'black_pct' not in _sub.columns and {'total_pop', 'black_pop'} <= set(_sub.columns):
            _sub['black_pct'] = (_sub.black_pop / _sub.total_pop * 100).round(2)
        _keep = [c for c in ['median_income', 'minority_pct', 'black_pct', 'total_pop', 'std_geoid']
                 if c in _sub.columns]
        gdf = gdf.merge(_sub[['account'] + _keep], on='account', how='left')
        print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
    except concurrent.futures.TimeoutError:
        print("Census API timed out — skipping census join")
        for _col in _census_cols:
            gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

In [ ]:
# Export — standardized per-parcel CSV
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

In [ ]:
# Parcel-map export + interactive HTML map (mappable parcels only)
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = gdf[gdf.geometry.notna()].copy()
map_gdf = save_parcel_map_export(
    gdf=map_gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | −1.72% vs CCRA 2025-26 commitment $42,944,985 (residual = condo/interleaved rows in the commitment-book PDF extraction) |
| Parcel count | 7,282 modeled taxable parcels (+222 fully-exempt held out) |
| Census coverage | 100% of mappable parcels matched to block groups |
| Geometry / map | 6,020 of 7,282 (82.7%) mappable; condo units share one footprint and are not individually mapped |
| Use-code coverage | 65.7% from VGSI; remainder inferred (vacant / homestead-residential / other-residential) |
| Artifact scan (Gate 5) | Clean — plausible building shares, no ceiling clustering, mobile homes correctly building-only |
| SFR median change | +3.5% (near break-even; large-lot suburban town) |
| Vacant land median change | +127% |
| Mobile Home median change | −43% (building-only, land owned by park) |

### Limitations
- **Combined levy:** Gorham commits municipal + school + county as one 14.800-mill bill; this models
  that full commitment (there is no separately-billed town-only rate to isolate).
- **Commitment-book parse residual (−1.72%):** ~20 interleaved condominium blocks in the source PDF
  are undercounted; the model is revenue-neutral on the parsed base.
- **Use-code inference (34% of parcels):** parcels outside VGSI's public coverage are classified from
  the commitment book's $0-building and Homestead/veteran flags; the "Other Residential" bucket
  (~13.6%) is inflated by this and should be read as "residential, subtype unknown."
- **Mapped subset (82.7%):** condominium units and some sublots lack a distinct polygon in the state
  layer and do not appear individually on the map, though they are in the tax model.
- **Legality:** illustrative only — Maine has no municipal split-rate authority (constitutional
  amendment required).